In [7]:
import pandas as pd
import numpy as np
import glob
import os

pathway = '/Users/kai_0514/Documents/Python/30y_Weather/'  # 替換為你的資料夾路徑
weather = ['Rainfall', 'Highest_Temp', 'Lowest_Temp', 'Avg_Temp']  # 替換為你的資料夾名稱列表
all_weather_data = {} # 用於存放所有讀取的資料

# 獲取該資料夾下所有 CSV 檔案的路徑

for w in weather:

    all_weather_data[w] = {}
    file_list = glob.glob(os.path.join(pathway, w, '*.csv'))  # 獲取每個資料夾下的 CSV 檔案路徑

    for file in file_list:

        df = pd.read_csv(file, encoding='utf-8')
        df = df.dropna(axis=1, how='all') # 刪除全為 NaN 的欄位
        df_long = df.melt(id_vars=['LON', 'LAT'], var_name='Date_Raw', value_name='Value') # 將資料轉換為長格式
        df_long.loc[df_long['Value'] <= -90, 'Value'] = np.nan # 將不合理的值替換為 NaN
        df_long['Date'] = pd.to_datetime(df_long['Date_Raw'], format='%Y%m', errors='coerce') # 將日期字串轉換為 datetime 格式，並處理無法解析的日期
        df_long['Date_adjust'] = df_long['Date'].dt.strftime('%Y-%m') # 將日期格式化為 "YYYY-MM"
        df_long['Year'] = df_long['Date'].dt.year # 從日期中提取年份
        df_long['Month'] = df_long['Date'].dt.month # 從日期中提取月份
        df_final = df_long.drop(columns=['Date_Raw', 'Date']) # 刪除原始日期欄位，保留格式化後的日期和年份、月份

        # 處理檔名：觀測_月資料_新北市_平均溫_1993.csv -> 擷取想要的 Key
        base_name = os.path.splitext(os.path.basename(file))[0]
        name_parts = base_name.split('_')
        data_key = f"{name_parts[2]}_{name_parts[-1]}"
        
        # 讀取並存入
        all_weather_data[w][data_key] = df_final

print("所有類別資料已讀取完畢！")

所有類別資料已讀取完畢！


In [8]:
for w in all_weather_data:
    append_list = []
    for key in all_weather_data[w]:
        append_list.append(all_weather_data[w][key])
    combined_df = pd.concat(append_list).sort_index()
    all_weather_data[w] = combined_df

In [57]:
all_weather_data['Rainfall']  # 示範查看某個資料表的前幾行

,LON,LAT,Value,Date_adjust,Year,Month
0,121.3,24.90,53.8384,1999-01,1999,1
0,121.3,24.90,86.7886,2013-01,2013,1
0,121.3,24.90,32.5190,2009-01,2009,1
0,121.3,24.90,194.5566,2012-01,2012,1
0,121.3,24.90,31.5086,2017-01,2017,1
...,...,...,...,...,...,...
1415,122.0,25.05,261.9643,2018-12,2018,12
1415,122.0,25.05,294.8672,1993-12,1993,12
1415,122.0,25.05,54.2614,1995-12,1995,12
1415,122.0,25.05,249.9163,2011-12,2011,12


In [9]:
import pandas as pd
import numpy as np

# 1. 取得年雨量總值 (1993-2022)
annual_rainfall = all_weather_data['Rainfall'].groupby(['LON', 'LAT', 'Year'])['Value'].sum().reset_index()
annual_avg_temp = all_weather_data['Avg_Temp'].groupby(['LON', 'LAT', 'Year'])['Value'].mean().reset_index()

# 2. 定義計算趨勢與平方差的函數
def calculate_trends(group):
    # 確保年份是連續排序的
    group = group[group['Value'].notna()]

    if len(group) < 2 or group['Value'].sum() == 0:
        return pd.Series({
            'Growth_Rate': np.nan, 
            'Total_Sq_Error': np.nan
        })
    
    # 自變數 X (年份), 應變數 Y (年雨量)
    x = group['Year'].values
    y = group['Value'].values
    
    # 使用 numpy 進行線性擬合 (y = ax + b)
    # deg=1 代表一次線性回歸
    slope, intercept = np.polyfit(x, y, deg=1)
    
    # 計算成長曲線預測值 (Predicted values)
    y_pred = slope * x + intercept
    
    # 計算平方差 (Residual Square Error)
    # 每個點與成長曲線的距離平方總和
    sq_error = np.sum((y - y_pred)**2)
    sq_error = round(sq_error, 0)  # 四捨五入到小數點後兩位
    
    # 回傳結果：成長率(斜率) 與 平方差
    return pd.Series({
        'Growth_Rate': slope,     # 每年雨量增減量 (mm/year)
        'Total_Sq_Error': sq_error # 平方差總和，越小代表降雨趨勢越穩定符合直線
    })

# 3. 針對每個座標點 (LON, LAT) 執行計算
rain_trend_results = annual_rainfall.groupby(['LON', 'LAT']).apply(calculate_trends, include_groups=False).reset_index()
temp_trend_results = annual_avg_temp.groupby(['LON', 'LAT']).apply(calculate_trends, include_groups=False).reset_index()

# 4. 查看結果
print(rain_trend_results)
print(temp_trend_results)

        LON    LAT  Growth_Rate  Total_Sq_Error
0    121.30  24.90    10.230573       7153795.0
1    121.30  24.95    15.136050       6674732.0
2    121.30  25.10    16.074973       6339691.0
3    121.30  25.15    16.070291       6600731.0
4    121.35  24.85    -2.224180      13343840.0
..      ...    ...          ...             ...
113  121.95  25.00     9.512258      14339104.0
114  121.95  25.05     7.370843      10757439.0
115  121.95  25.10          NaN             NaN
116  122.00  25.00    -3.792849       7118334.0
117  122.00  25.05     1.939233       6991645.0

[118 rows x 4 columns]
        LON    LAT  Growth_Rate  Total_Sq_Error
0    121.30  24.90     0.020105             4.0
1    121.30  24.95     0.031295             4.0
2    121.30  25.10     0.026667             4.0
3    121.30  25.15     0.026344             4.0
4    121.35  24.85     0.022782             4.0
..      ...    ...          ...             ...
113  121.95  25.00     0.015950             2.0
114  121.95  25.

In [10]:
# 1. 取得最高溫與最低溫的原始資料 (長表格)
# 假設你之前的 all_weather_data 已經清好了
df_high = all_weather_data['Highest_Temp']
df_low = all_weather_data['Lowest_Temp']

# 2. 合併資料表
# 透過經緯度、年份、月份進行對齊
df_range = pd.merge(df_high, df_low, on=['LON', 'LAT', 'Year', 'Month'], suffixes=('_H', '_L'))

# 3. 計算月溫差 (Value_H - Value_L)
df_range['Monthly_Range'] = df_range['Value_H'] - df_range['Value_L']

# 4. 取得「年平均月溫差」
# 這樣可以觀察 30 年來，每年的溫差是在擴大還是縮小
annual_temp_range = df_range.groupby(['LON', 'LAT', 'Year'])['Monthly_Range'].mean().reset_index()

# 為了統一變數名稱進入 function，我們把 Monthly_Range 改名回 Value
annual_temp_range = annual_temp_range.rename(columns={'Monthly_Range': 'Value'})

In [11]:
# 使用你定義好的 function 進行計算
range_trend_results = annual_temp_range.groupby(['LON', 'LAT']).apply(
    calculate_trends, include_groups=False
).reset_index()

# 查看結果
print("--- 月溫差趨勢分析 ---")
print(range_trend_results.head())

--- 月溫差趨勢分析 ---
      LON    LAT  Growth_Rate  Total_Sq_Error
0  121.30  24.90    -0.023260             4.0
1  121.30  24.95    -0.020205             4.0
2  121.30  25.10    -0.036673             4.0
3  121.30  25.15    -0.033578             4.0
4  121.35  24.85    -0.018777             3.0


In [59]:
import geopandas as gpd
from shapely.geometry import Point
import os

# 設定輸出路徑
output_dir = os.path.join(pathway, 'SHP_Outputs')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 定義一個轉換並輸出 SHP 的小工具
def export_to_shp(df, column_name, file_label):
    # 1. 建立幾何物件 (Point)
    geometry = [Point(xy) for xy in zip(df['LON'], df['LAT'])]
    
    # 2. 建立 GeoDataFrame
    # 我們只保留經緯度跟目標數值，並設定座標系統為 WGS84 (EPSG:4326)
    gdf = gpd.GeoDataFrame(df[['LON', 'LAT', column_name]], geometry=geometry, crs="EPSG:4326")
    
    # 3. 輸出檔案 (注意：SHP 欄位名會自動截斷，建議保持簡短)
    file_path = os.path.join(output_dir, f"{file_label}.shp")
    gdf.to_file(file_path, encoding='utf-8')
    print(f"已匯出：{file_path}")

In [ ]:
from sqlalchemy import create_engine
import geopandas as gpd
from shapely.geometry import Point

# 1. 載入秘密檔案
load_dotenv()

# 2. 從環境變數中讀取網址，如果找不到就給一個預設值或報錯
# 註解：這樣 GitHub 上的代碼就不會看到你的帳號名稱了！
db_url = os.getenv("DB_URL")
if not db_url:
    raise ValueError("找不到資料庫連線資訊，請檢查 .env 檔案！")

engine = create_engine(db_url)
def export_to_postgis(df, column_name, table_name):
    """
    將 DataFrame 直接存入 PostGIS 資料庫
    """
    # 建立幾何物件
    geometry = [Point(xy) for xy in zip(df['LON'], df['LAT'])]
    
    # 建立 GeoDataFrame (設定座標系統為 WGS84)
    # 這裡可以保留所有欄位，不需要像 SHP 那樣擔心欄位名稱太長
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
    
    # 2. 寫入 PostGIS
    # if_exists='replace' 表示如果資料表已存在就覆蓋它
    # index=False 不把 DataFrame 的索引當作欄位存入
    gdf.to_postgis(name=table_name, con=engine, if_exists='replace', index=False)
    
    print(f"✅ 資料已成功存入資料庫表：{table_name}")

# 使用範例：
# 假設你的氣象分析結果在 df_results 裡，想存成名為 'rainfall_analysis' 的資料表
# export_to_postgis(df_results, 'RAINFALL', 'rainfall_analysis')

In [12]:
# --- A. 平均溫系列 ---
# 將整個 temp_trend_results 存入，這樣 Growth_Rate 和 SqError 都會在同一個表裡
export_to_postgis(temp_trend_results, 'Growth_Rate', 'temp_trend_analysis')

# --- B. 年雨量系列 ---
export_to_postgis(rain_trend_results, 'Growth_Rate', 'rain_trend_analysis')

# --- C. 溫差系列 ---
export_to_postgis(range_trend_results, 'Growth_Rate', 'range_trend_analysis')

print("\n--- 所有分析結果已成功匯入 PostGIS 資料庫：30y_weather_db ---")

✅ 資料已成功存入資料庫表：temp_trend_analysis
✅ 資料已成功存入資料庫表：rain_trend_analysis
✅ 資料已成功存入資料庫表：range_trend_analysis

--- 所有分析結果已成功匯入 PostGIS 資料庫：30y_weather_db ---


In [60]:
# --- A. 平均溫系列 ---
export_to_shp(temp_trend_results, 'Growth_Rate', 'AvgTemp_GrowthRate')
export_to_shp(temp_trend_results, 'Total_Sq_Error', 'AvgTemp_SqError')

# --- B. 年雨量系列 ---
export_to_shp(rain_trend_results, 'Growth_Rate', 'Rainfall_GrowthRate')
export_to_shp(rain_trend_results, 'Total_Sq_Error', 'Rainfall_SqError')

# --- C. 溫差系列 ---
export_to_shp(range_trend_results, 'Growth_Rate', 'TempRange_GrowthRate')
export_to_shp(range_trend_results, 'Total_Sq_Error', 'TempRange_SqError')

print("\n--- 所有 SHP 檔案已輸出至 SHP_Outputs 資料夾 ---")

已匯出：/Users/kai_0514/Documents/Python/30y_Weather/SHP_Outputs/AvgTemp_GrowthRate.shp
已匯出：/Users/kai_0514/Documents/Python/30y_Weather/SHP_Outputs/AvgTemp_SqError.shp
已匯出：/Users/kai_0514/Documents/Python/30y_Weather/SHP_Outputs/Rainfall_GrowthRate.shp
已匯出：/Users/kai_0514/Documents/Python/30y_Weather/SHP_Outputs/Rainfall_SqError.shp
已匯出：/Users/kai_0514/Documents/Python/30y_Weather/SHP_Outputs/TempRange_GrowthRate.shp
已匯出：/Users/kai_0514/Documents/Python/30y_Weather/SHP_Outputs/TempRange_SqError.shp

--- 所有 SHP 檔案已輸出至 SHP_Outputs 資料夾 ---


/var/folders/17/vsmmd5dx5pv74fwy86w422940000gn/T/ipykernel_58142/2988947315.py:21: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(file_path, encoding='utf-8')
/Users/kai_0514/Library/Python/3.9/lib/python/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Growth_Rate' to 'Growth_Rat'
  ogr_write(
/var/folders/17/vsmmd5dx5pv74fwy86w422940000gn/T/ipykernel_58142/2988947315.py:21: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(file_path, encoding='utf-8')
/Users/kai_0514/Library/Python/3.9/lib/python/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Total_Sq_Error' to 'Total_Sq_E'
  ogr_write(
/var/folders/17/vsmmd5dx5pv74fwy86w422940000gn/T/ipykernel_58142/2988947315.py:21: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(file